# Mały projekt 1: wizualizacja poziomu zanieczyszczeń (PM2.5)

Główny Inspektorat Ochrony Środowiska (GIOS) udostępnia dane o jakości powietrza w Polsce na stronie [https://powietrze.gios.gov.pl](https://powietrze.gios.gov.pl), tj. poziom stężenia pyłów PM2.5, PM10, SO2 i innych zanieczyszczeń. Dane te są szczególnie przydatne w analizach środowiskowych i zdrowotnych. W tym zadaniu interesują nas godzinne pomiary stężeń drobnego pyłu **PM2.5** (pyłu o średnicy poniżej 2.5 µm) w latach **2014, 2019 i 2024**. Pyły PM2.5 są one bardzo szkodliwe dla zdrowia, gdyż mogąc przenikać głęboko do układu oddechowego i krwiobiegu. Zadanie polega na przeprowadzeniu analizy danych wraz z opisami czynności oraz wykresami.

## Wymagania ogólne

- Rozwiązanie należy przygotować w formie **Jupyter Notebooka (`.ipynb`)**.  Rozwiązanie proszę przesłać przez Moodle’a.
- Wszystkie operacje czyszczenia i łączenia danych wykonaj **programistycznie**, bez ręcznej edycji plików.
- Do każdego punktu należy obowiązkowo dodać opis z analizą otrzymanych wyników.  
- Zadanie należy wykonać w wylosowanych parach. Obie osoby z pary przesyłają **identyczne pliki** i dodają na początku notebooka opis swojego wkładu.
- Ocena jest wspólna dla pary, a nie indywidualna.

## Dane wejściowe i metadane

- Każdy rok to archiwum ZIP dostępne pod adresem [https://powietrze.gios.gov.pl/pjp/archives/](https://powietrze.gios.gov.pl/pjp/archives/).
- Wewnątrz znajdują się pliki Excel, np. `2024_PM25_1g.xlsx`.
- Poniższa funkcja `download_gios_archive` pozwala odczytać odpowiedni zbior danych; resztę danych pomijamy.
- Użyj pliku metadanych (dostępnego na tej samej stronie), aby zaktualizować **kody stacji** (część stacji mogła w międzyczasie zmienić nazwę). W metadanych znajdziesz odpowiednie kolumny.

## Zadania

### 1. Wczytanie i czyszczenie danych

Wczytaj dane dla lat **2014, 2019 i 2024**, oczyścić je z niepotrzebnych wierszy oraz ujednolić ich format. Zaktualizuj stare kody stacji zgodnie z metadanymi. Pozostaw tylko stacje występujące we wszystkich trzech latach. Informacje o stacjach pomiarowych warto uzupełnić o miejscowości dostępne w metadanych, np. za pomocą MultiIndex: (miejscowość, kod stacji). Pomiary dokonane o północy (00:00:00) powinny być potraktowane jako dotyczące poprzedniego dnia. Połącz dane z trzech lat w jeden `DataFrame` i zapisz do pliku.

### 2. Średnie miesięczne + trend dla miast

Oblicz średnie miesięczne stężenie PM2.5 dla każdej stacji i roku. Dla **Warszawy** i **Katowic**, po uśrednieniu po wszystkich stacjach z tych miast, narysuj wykres liniowy pokazujący trend średnich miesięcznych wartości PM2.5 w 2014 i 2024 roku. Oś X - miesiące (1-12); oś Y - średnia wartość PM2.5; 4 linie trendu. Dołącz opis i interpretację obserwowanych różnic.

### 3. Heatmapa miesięcznych średnich

Dla każdej miejscowości przedstaw heatmapę średnich miesięcznych stężeń PM2.5 w latach lat 2014, 2019 i 2024 (oś X – miesiąc, oś Y – rok). Uśrednij wartości po wszystkich stacjach w danej miejscowości. Każdy panel (facet) ma odpowiadać jednej miejscowości. Dołącz interpretację obserwowanych wyników.

### 4. Dni z przekroczeniem normy (WHO)

Dla każdej stacji i roku policz liczbę dni, w których wystąpiło przekroczenie dobowej normy stężenia PM2.5, czyli 15 µg/m³ (źródło: [https://airscan.org/new-who-air-quality-guidelines-2021/](https://airscan.org/new-who-air-quality-guidelines-2021/)). Znajdź 3 stacje z najmniejszą i 3 stacje z największą liczbą dni z przekroczeniem normy dobowej w 2024 roku. Dla tych 6 stacji narysuj *grouped barplot*, gdzie oś X – stacje, oś Y – liczba dni z przekroczeniem, kolor – rok (2014, 2019, 2024). Dołącz opis i interpretację obserwowanych różnic.

## Dodatkowe wymagania i sugestie

- Notebook powinien zawierać *sanity checks*, np.:
  - liczba stacji w każdym pliku,
  - liczba dni w każdym roku,  
  - kilka przykładowych mapowań kodów stacji,  
- Wszystkie wykresy powinny mieć tytuły, legendy i krótki opis interpretacji.
- Zachęcamy do weryfikacji kodu napisanego przez drugą osobę, gdyż ocena jest wspólna.
- Można wykorzystać dowolne poznane biblioteki do analizy i wizualizacji danych w Pythonie.

## Kryteria oceny

- Zadanie 1: 3 pkt
- Zadanie 2: 2 pkt
- Zadanie 3: 1.5 pkt
- Zadanie 4: 2 pkt
- Jakość wyjaśnień, interpretacje, opis wkładu: 1.5 pkt



In [ ]:
from pathlib import Path
import pandas as pd
import requests
import zipfile
import io, os


# id archiwum dla poszczególnych lat
gios_archive_url = "https://powietrze.gios.gov.pl/pjp/archives/downloadFile/"
gios_url_ids = {2014: '302', 2019: '322', 2024: '582'}
gios_pm25_file = {2014: '2014_PM2.5_1g.xlsx', 2019: '2019_PM25_1g.xlsx', 2024: '2024_PM25_1g.xlsx'}

# funkcja do ściągania podanego archiwum
def download_gios_archive(year, gios_id, filename):
    # Pobranie archiwum ZIP do pamięci
    #TODO propozycja: ja bym proponował zrobione tego za pomocą path'a, bo jest czytelniej
    url = f"{gios_archive_url}{gios_id}"
    response = requests.get(url)
    response.raise_for_status()  # jeśli błąd HTTP, zatrzymaj
    
    # Otwórz zip w pamięci
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        # znajdź właściwy plik z PM2.5
        #TODO 
        if not filename:
            print(f"Błąd: nie znaleziono {filename}.")
        else:
            # wczytaj plik do pandas
            with z.open(filename) as f:
                try:
                    df = pd.read_excel(f, header=None)
                except Exception as e:
                    print(f"Błąd przy wczytywaniu {year}: {e}")
    return df



In [ ]:
def clean_gios_data(df_raw: pd.DataFrame)->pd.DataFrame:
    """
    Funkcja służy do oczyszczania danych pomiarowych PM2.5 pochodzących z archiwów GIOŚ.
    Zwracany jest obiekt DataFrame z indeksem czasowym i kolumnami odpowiadającymi stacjom pomiarowym.
    """
   
    
    # Usunięcie wierszy całkowicie pustych
    df = df_raw.dropna(how='all')

    # Wykrycie pierwszego wiersza zawierającego datę 
    date_pattern = r'^\d{4}-\d{2}-\d{2}'
    start_idx = df[df.iloc[:, 0].astype(str).str.match(date_pattern, na=False)].index[0]

    # Ustawienie nazw kolumn na podstawie wiersza poprzedzającego dane (zawierającego nazwy stacji)
    df.columns = df.iloc[start_idx - 1]

    # Zachowanie tylko wierszy z rzeczywistymi pomiarami
    df = df.iloc[start_idx:].copy()

    # Zmiana nazwy pierwszej kolumny na „Data”, która zawiera daty pomiarów
    df.rename(columns={df.columns[0]: "Data"}, inplace=True)

    # Konwersja kolumny „Data” na typ datetime w celu poprawnej interpretacji czasu
    df["Data"] = pd.to_datetime(df["Data"], errors="coerce", format="%Y-%m-%d %H:%M:%S")

    # Ustawienie kolumny „Data” jako indeksu czasowego
    df.set_index("Data", inplace=True)

    # Konwersja pozostałych kolumn na wartości liczbowe
    df = df.apply(pd.to_numeric, errors="coerce")

    # Korekta godzin 00:00 – pomiary o północy przypisywane do poprzedniego dnia
    mask_midnight = df.index.hour == 0
    df.index = df.index - pd.to_timedelta(mask_midnight.astype(int), unit="D")

    return df


In [ ]:
df2014_raw = download_gios_archive(2014, gios_url_ids[2014], gios_pm25_file[2014])
df2019_raw = download_gios_archive(2019, gios_url_ids[2019], gios_pm25_file[2019])
df2024_raw = download_gios_archive(2024, gios_url_ids[2024], gios_pm25_file[2024])

#df2014 = clean_gios_data(df2014_raw)
df2019 = clean_gios_data(df2019_raw)
df2024 = clean_gios_data(df2024_raw)


In [ ]:
df2019

In [ ]:
# Pierwszy wiersz ustawiony jako nagłówki kolumn  
df2014_raw.columns = df2014_raw.iloc[0]
# Usunięcie pierwszego wiersza, który został wykorzystany jako nagłówek  
df2014 = df2014_raw.drop(index=0).reset_index(drop=True)
# Zmiana nazwy pierwszej kolumny na „Data”, ponieważ zawiera daty pomiarów  
df2014.rename(columns={df2014.columns[0]: "Data"}, inplace=True)
# Usunięcie kolumn zawierających wyłącznie wartości puste  
df2014 = df2014.dropna(axis=1, how='all')
# Konwersja kolumny „Data” na typ datetime w celu umożliwienia operacji na datach  
df2014["Data"] = pd.to_datetime(df2014["Data"], errors="coerce")
# Ustawienie kolumny „Data” jako indeksu tabeli  
df2014.set_index("Data", inplace=True)
# Konwersja pozostałych kolumn (stacji pomiarowych) na wartości liczbowe  
df2014 = df2014.apply(pd.to_numeric, errors="coerce")
df2014 = df2014.iloc[2:]
df2014.head()

In [ ]:
# TODO - to można dać do wcześniejszych funkcji, 
meta = pd.read_excel("https://powietrze.gios.gov.pl/pjp/archives/downloadFile/584", header=0)
meta["Stary Kod stacji \n(o ile inny od aktualnego)"]

In [ ]:
mapping = {}
for _, row in meta.dropna(subset=["Stary Kod stacji \n(o ile inny od aktualnego)"]).iterrows():
    old_codes = [c.strip() for c in row["Stary Kod stacji \n(o ile inny od aktualnego)"].split(",")]
    for old in old_codes:
        mapping[old] = row["Kod stacji"]
mapping

In [ ]:
def clean_column_names(df):
    """""Czyszczenie nazw kolum w celu ujednolicenia ich pomiędzy różnymi latami danych i ułatwienia mapowania."""
    df.columns = (
        df.columns.str.strip()
        .str.replace("-PM2.5-1g", "", regex=False)
        .str.replace(" ", "")
    )
    return df

df2019 = clean_column_names(df2019)
df2024 = clean_column_names(df2024)

print(df2014.columns[:10])
print(df2019.columns[:10])
print(df2024.columns[:10])


In [ ]:
df2014.rename(columns=mapping, inplace=True)
df2019.rename(columns=mapping, inplace=True)
df2024.rename(columns=mapping, inplace=True)


In [ ]:
common_stations = list(set(df2014.columns) & set(df2019.columns) & set(df2024.columns))
print(f"Liczba wspólnych stacji: {len(common_stations)}")

df2014 = df2014[common_stations]
df2019 = df2019[common_stations]
df2024 = df2024[common_stations]
df2014.head()


In [ ]:
combined = pd.concat([df2014, df2019, df2024]).sort_index()
combined["MpKrakBulwar"]

In [ ]:
combined = combined.dropna(how="all")

combined.index = pd.to_datetime(combined.index)
combined = combined.sort_index()

print(combined.shape)
print(combined.index.min(), combined.index.max())
#TODO - funkcje podmieniłem tutaj się nie wyświetli bo jest niżej
combined = fill_na(combined)
combined.to_csv("PM25_combined_2014_2019_2024.csv")


In [ ]:
df = combined
df

## Zadanie 2
### Metodologia 

>Średnie miesięczne

Do wyliczenia średnich miesięcznych używamy metody:
```Python
time_to_month
```
Otrzymujemy wtedy `df` indeksowany po pierwszym dniu miesiąca, która zawiera średnią wartości emisji z miesiąca zmierzonej w danej stacji.

> 

### Zadanie 2
#### Wyniki
Widzimy, że w 2014 roku stężenie PM2.5 jest około dwa razy większe niż w roku 2024. 

Możemy także zauważyć ....

TODO 
#### Analiza wyników
Wynika to z tego, że w tych latach ... (napisać co zostało zmienione z czego to wynika) 
#### Zadanie 2


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# helper functions 
def fill_na(df: pd.DataFrame) ->pd.DataFrame:
    '''
    Fills NA values in df by dropping empty values
    #TODO 
    - stworzenie funkcji
    '''
    return df.dropna()


def time_to_month(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Changes time index representation from day to month, by mean aggregating. Returns new df
    \n**df**: Dataframe with time indexes
    '''
    new_df = df.resample('MS').mean()
    return new_df

# musimy mieć funkcje która zbiera województwo i miejscowość

def get_stations(df: pd.DataFrame, cities: list, *, meta=meta) -> list:
    '''
    Return df with columns that met criteria
    \n**df**: Dataframe based on GIOŚ data TODO - tutaj zmienic nazwę po funkcji która te dane przetwarza 
    '''
    filt = meta['Miejscowość'].isin(cities)
    # filt1 = meta['Województwo'].isin(voivodeships)
    # filt = filt1 & filt
    cols_idx = meta[filt]['Kod stacji'].to_list()
    
    df_filtered = df.loc[:, df.columns.isin(cols_idx)]
    return df_filtered

def years_trend_cities(df: pd.DataFrame, cities: list, years: list, *, meta=meta, save=False) -> None:
    '''
    Draw plots how concentration of PM2.5 was changing in given year across different cities
    \n**df**: Dataframe based on GIOŚ data TODO - tutaj zmienic nazwę po funkcji 
    \n**cities**: List of cities to compare 
    \n**years**: List of years we want to compare data
    \n**meta**: Dataframe with meta information about **df**
    \n**save**: TODO If False return None, else save plots in given folder 
    '''
    # Changes time representation: per day -> per month (mean)
    df = time_to_month(df)
    
    # Changes all to all cities
    if cities == 'all':
        # finding all cities in df
        cities = meta[meta['Kod stacji'].isin(df.columns)]['Miejscowość'].to_list()
        cities = list(set(cities))
        print(f"There is {len(cities)}")

    # plots
    for year in years:
        df_loc = df[df.index.year == year]
        for city in cities:
            # Gets stations which met condition
            values = get_stations(df_loc, cities=[city], meta=meta).mean(axis=1)
            # agg for every year 
            if not values.isna().any():
                values.plot(label=city)
        plt.legend()
        plt.title(f'Mean PM2.5 concentration in {year}')
        plt.xlabel('Month')
        plt.ylabel('Concentration of PM2.5')
        plt.ylim((0, df.mean(axis=1).max()*1.25))
        plt.show()

def years_heatmaps_cities(df: pd.DataFrame, cities: list, years: list, *, meta=meta, save=False) -> None:
    '''
    Draw heatmaps Months $\\times$ Years which shows concentration pf PM.2.5
    \n**df**: Dataframe based on GIOŚ data TODO - tutaj zmienic nazwę po funkcji 
    \n**cities**: List of cities to compare 
    \n**years**: List of years we want to compare data
    \n**meta**: Dataframe with meta information about **df**
    \n**save**: TODO If False return None, else save plots in given folder 
    '''
    # Changes time representation: per day -> per month (mean)
    df = df[df.index.year.isin(years)]
    
    df = time_to_month(df)
    # Changes all to all cities
    if cities == 'all':
        # finding all cities in df
        cities = meta[meta['Kod stacji'].isin(df.columns)]['Miejscowość'].to_list()
        cities = list(set(cities))
        print(f"There is {len(cities)}")

   # plots 
    for city in cities:
        val_col = f"city_mean"
        # Gets stations which met condition
        df[val_col] = get_stations(df, cities=[city], meta=meta).mean(axis=1)
        if not df[val_col].isna().any():
            continue 
        # creating pivot table for sns.heatmap
        pivot = pd.pivot_table(df, val_col,df.index.year, df.index.month, aggfunc='mean')
        # plotting pivot table
        sns.heatmap(pivot)
        plt.title(f'Mean PM2.5 concentration in {city}')
        plt.xlabel('Month')
        plt.ylabel('Year')
        #TODO skale ustawić wszędzie taką samą i inny kolor, dostosował bym ją tak, że pokazuje gdzie jest przekroczony poziom zagrażający życiu 
        plt.show()

In [ ]:
# Exercise 2
cities = ['Warszawa', 'Katowice']
years = [2014, 2024]
years_trend_cities(df, cities=cities, years=years)

In [ ]:
# Exercise 3
cities = ['Warszawa', 'Katowice']
years = [2014, 2019, 2024]
years_heatmaps_cities(df, cities='all', years=years)